In [ ]:
!pip install cartopy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.colors import ListedColormap, BoundaryNorm
import ee
import geemap
from ee import EEException # Explicitly import EEException

# Study domain coordinates
lat_min, lat_max = 5.04, 38.52
lon_min, lon_max = 65.04, 98.52


# Earth Engine Authentication and Initialization
# Explicitly authenticate and initialize to ensure state after kernel restarts
print("Attempting Earth Engine authentication and initialization...")
try:
    ee.Authenticate() # This may require interactive user input (browser window)
    ee.Initialize(project = 'souvagya')
    print("Earth Engine initialized successfully for project 'souvagya'.")
except EEException as e:
    print(f"EE initialization failed: {e}. Please ensure you have authenticated and configured your project 'souvagya'.")
    raise # Re-raise the exception if initialization truly failed

# Define the region of interest
roi = ee.Geometry.Rectangle([lon_min, lat_min, lon_max, lat_max])


In [ ]:
# =============================================================================
# Fetch and Process Elevation Data (SRTM) - RAW
# =============================================================================
srtm = ee.Image('USGS/SRTMGL1_003')
elevation = srtm.select('elevation').clip(roi)
elevation_raw = geemap.ee_to_numpy(elevation, region=roi, scale=1000)

# Process raw elevation data for plotting
elevation_raw_processed = np.flipud(elevation_raw.squeeze()).astype(float)
elevation_raw_processed[elevation_raw_processed == -32767] = np.nan

In [ ]:
# =============================================================================
# Fetch and Process Land Use/Land Cover (MODIS) - RAW
# =============================================================================
lulc_dataset = ee.ImageCollection('MODIS/006/MCD12Q1')
lulc = lulc_dataset.filterDate('2020-01-01', '2020-12-31').first()
lulc_img = lulc.select('LC_Type1').clip(roi)

# Reproject to EPSG:4326 before converting to numpy (from previous fix)
lulc_img_reproj = lulc_img.reproject(
    crs='EPSG:4326',
    scale=1000  # Keep 1km resolution
)
lulc_raw = geemap.ee_to_numpy(lulc_img_reproj, region=roi, scale=1000)

# Process raw LULC data for plotting
lulc_raw_processed = np.flipud(lulc_raw.squeeze())

In [ ]:
# =============================================================================
# Fetch and Process Mean Temperature (ERA5) - RAW
# =============================================================================
era5 = ee.ImageCollection('ECMWF/ERA5_LAND/DAILY_AGGR')
temp_collection = era5.filterDate('2020-01-01', '2020-12-31').select('temperature_2m').mean().clip(roi)
temp_raw = geemap.ee_to_numpy(temp_collection, region=roi, scale=10000)

# Process raw temperature data for plotting
temp_raw_processed = np.flipud(temp_raw.squeeze())

In [ ]:
# Start plotting
fig = plt.figure(figsize=(20, 16))
plt.subplots_adjust(wspace=0.3, hspace=0.35)
projection = ccrs.PlateCarree()

# -----------------------------------------------------------------------------
# Panel (a): Location Map
# -----------------------------------------------------------------------------
ax1 = plt.subplot(2, 2, 1, projection=projection)

# Set extent for South Asia context
context_extent = [50, 110, 0, 45]
ax1.set_extent(context_extent, crs=projection)

# Add map features
# ax1.add_feature(cfeature.LAND, facecolor='lightgray', alpha=0.5)
# ax1.add_feature(cfeature.OCEAN, facecolor='lightblue', alpha=0.3)
ax1.add_feature(cfeature.COASTLINE, linewidth=1.0, edgecolor='black')
# ax1.add_feature(cfeature.BORDERS, linewidth=0.8, linestyle='--', edgecolor='gray')
# ax1.add_feature(cfeature.LAKES, facecolor='lightblue', alpha=0.5)

# Add gridlines
gl1 = ax1.gridlines(draw_labels=True, dms=False, x_inline=False, y_inline=False,
                    linewidth=0.5, alpha=0.9, linestyle='--', color='gray')
gl1.top_labels = False
gl1.right_labels = False
gl1.xlabel_style = {'size': 12, 'weight': 'bold'}
gl1.ylabel_style = {'size': 12, 'weight': 'bold'}

# Highlight study domain with red box
study_box = mpatches.Rectangle((lon_min, lat_min), lon_max - lon_min, lat_max - lat_min,
                               linewidth=3, edgecolor='red', facecolor='red',
                               alpha=0.3, transform=projection, zorder=10)
ax1.add_patch(study_box)

# Add labels
# ax1.text(77, 22, 'INDIA', transform=projection, fontsize=14,
#         fontweight='bold', ha='center', style='italic')
# ax1.text(80, 8, 'Indian Ocean', transform=projection, fontsize=10,
#         ha='center', style='italic', alpha=0.7, color='navy')

# Add domain info
# domain_info = f'{lat_min}°N - {lat_max}°N\n{lon_min}°E - {lon_max}°E\n{elevation_raw_processed.shape[0]}x{elevation_raw_processed.shape[1]} raw grid'
# props = dict(boxstyle='round', facecolor='white', alpha=0.9, edgecolor='red', linewidth=2)
# ax1.text(0.98, 0.02, domain_info, transform=ax1.transAxes, fontsize=10,
#         verticalalignment='bottom', horizontalalignment='right', bbox=props,
#         fontweight='bold', color='red')

ax1.set_title('(a) Study Domain Location', fontsize=13, fontweight='bold', pad=12)

# -----------------------------------------------------------------------------
# Panel (b): Raw Elevation
# -----------------------------------------------------------------------------
ax2 = plt.subplot(2, 2, 2, projection=projection)
ax2.set_extent([lon_min, lon_max, lat_min, lat_max], crs=projection)

# Add basic features
# ax2.add_feature(cfeature.COASTLINE, linewidth=0.8, edgecolor='black')

# Define raw coordinate grids specific to elevation data
raw_lats_elev = np.linspace(lat_min, lat_max, elevation_raw_processed.shape[0])
raw_lons_elev = np.linspace(lon_min, lon_max, elevation_raw_processed.shape[1])

# Plot raw elevation
im2 = ax2.contourf(raw_lons_elev, raw_lats_elev, elevation_raw_processed, levels=20, cmap='jet',
                   transform=projection, extend='both')

# Add contour lines
contours = ax2.contour(raw_lons_elev, raw_lats_elev, elevation_raw_processed, levels=10, colors='black',
                      linewidths=0.5, alpha=0.3, transform=projection)
ax2.clabel(contours, inline=True, fontsize=7, fmt='%d m')

# Gridlines
gl2 = ax2.gridlines(draw_labels=True, dms=False, x_inline=False, y_inline=False,
                    linewidth=0.5, alpha=0.9, linestyle='--', color='gray')
gl2.top_labels = False
gl2.right_labels = False
gl2.xlabel_style = {'size': 12, 'weight': 'bold'}
gl2.ylabel_style = {'size': 12, 'weight': 'bold'}

# Colorbar
cbar2 = plt.colorbar(im2, ax=ax2, orientation='vertical', pad=0.03, shrink=0.9)
cbar2.set_label('Elevation (m)', fontsize=10, fontweight='bold')
cbar2.ax.tick_params(labelsize=10)
# Make tick labels bold
for label in cbar2.ax.get_yticklabels():
    label.set_fontweight('bold')

# Statistics
# elev_stats_raw = f'Min: {np.nanmin(elevation_raw_processed):.0f} m\nMax: {np.nanmax(elevation_raw_processed):.0f} m\nMean: {np.nanmean(elevation_raw_processed):.0f} m'
# props2 = dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor='black')
# ax2.text(0.02, 0.98, elev_stats_raw, transform=ax2.transAxes, fontsize=9,
#         verticalalignment='top', bbox=props2)

ax2.set_title('(b) Topography (SRTM)', fontsize=13, fontweight='bold', pad=12)

# -----------------------------------------------------------------------------
# Panel (c): Raw Land Use/Land Cover
# -----------------------------------------------------------------------------
ax3 = plt.subplot(2, 2, 3, projection=projection)
ax3.set_extent([lon_min, lon_max, lat_min, lat_max], crs=projection)

# Add basic features
# ax3.add_feature((cfeature.COASTLINE), linewidth=0.8, edgecolor='black')

# Define raw coordinate grids specific to LULC data - MODIFIED for cell edges
# Assuming lon_min/max, lat_min/max define the overall bounds, and pixels are centered within them
dx_lulc = (lon_max - lon_min) / lulc_raw_processed.shape[1]
dy_lulc = (lat_max - lat_min) / lulc_raw_processed.shape[0]
raw_lons_lulc = np.linspace(lon_min - dx_lulc/2, lon_max + dx_lulc/2, lulc_raw_processed.shape[1] + 1)
raw_lats_lulc = np.linspace(lat_min - dy_lulc/2, lat_max + dy_lulc/2, lulc_raw_processed.shape[0] + 1)

# Define a comprehensive mapping for MODIS IGBP land cover classes
modis_lulc_mapping = {
    0: {'color': '#1c0dff', 'label': 'Water'},
    1: {'color': '#05450a', 'label': 'Evergreen Needleleaf'},
    2: {'color': '#086a10', 'label': 'Evergreen Broadleaf'},
    3: {'color': '#54a708', 'label': 'Deciduous Needleleaf'},
    4: {'color': '#78d203', 'label': 'Deciduous Broadleaf'},
    5: {'color': '#009900', 'label': 'Mixed Forest'},
    6: {'color': '#c6b044', 'label': 'Closed Shrublands'},
    7: {'color': '#dcd159', 'label': 'Open Shrublands'},
    8: {'color': '#dade48', 'label': 'Woody Savannas'},
    9: {'color': '#fbff13', 'label': 'Savannas'},
    10: {'color': '#b6ff05', 'label': 'Grasslands'},
    11: {'color': '#27ff87', 'label': 'Permanent Wetlands'},
    12: {'color': '#c24f44', 'label': 'Croplands'},
    13: {'color': '#a5a5a5', 'label': 'Urban'},
    14: {'color': '#ff6d4c', 'label': 'Cropland/Natural Veg'},
    15: {'color': '#69fff8', 'label': 'Snow/Ice'},
    16: {'color': '#f9ffa4', 'label': 'Barren'},
    17: {'color': '#1c0dff', 'label': 'Water'},
    255: {'color': '#FFFFFF', 'label': 'No Data/Unclassified'} # MODIS fill value
}

# Extract unique_values from lulc_raw_processed, handling NaN values
unique_values_raw_lulc = np.unique(lulc_raw_processed[~np.isnan(lulc_raw_processed)]).astype(int)

# Create filtered_colors and filtered_labels based on unique_values
filtered_colors_raw = [modis_lulc_mapping[val]['color'] for val in unique_values_raw_lulc if val in modis_lulc_mapping]
filtered_labels_raw = [modis_lulc_mapping[val]['label'] for val in unique_values_raw_lulc if val in modis_lulc_mapping]

# Create custom colormap
cmap_lulc_raw = ListedColormap(filtered_colors_raw)

# Create BoundaryNorm
bounds_raw_lulc = np.concatenate([unique_values_raw_lulc - 0.5, [unique_values_raw_lulc.max() + 0.5]])
norm_raw_lulc = BoundaryNorm(bounds_raw_lulc, cmap_lulc_raw.N)

# Plot raw LULC - MODIFIED for shading='flat'
im3 = ax3.pcolormesh(raw_lons_lulc, raw_lats_lulc, lulc_raw_processed, cmap=cmap_lulc_raw, norm=norm_raw_lulc,
                     transform=projection, shading='flat')

# Gridlines
gl3 = ax3.gridlines(draw_labels=True, dms=False, x_inline=False, y_inline=False,
                    linewidth=0.5, alpha=0.9, linestyle='--', color='gray')
gl3.top_labels = False
gl3.right_labels = False
gl3.xlabel_style = {'size': 12,'weight': 'bold'}
gl3.ylabel_style = {'size': 12,'weight': 'bold'}

# Colorbar with labels
cbar3 = plt.colorbar(im3, ax=ax3, orientation='vertical', pad=0.03, shrink=0.9)
cbar3.set_label('Land Cover Type', fontsize=12, fontweight='bold')

# Calculate midpoints for tick labels
midpoints = (bounds_raw_lulc[:-1] + bounds_raw_lulc[1:]) / 2
cbar3.set_ticks(midpoints)
cbar3.set_ticklabels(filtered_labels_raw, fontsize=10, fontweight='bold')
cbar3.ax.minorticks_off()

ax3.set_title('(c) Land Use/Land Cover (MODIS IGBP)', fontsize=13, fontweight='bold', pad=12)

# -----------------------------------------------------------------------------
# Panel (d): Raw Mean Temperature
# -----------------------------------------------------------------------------
ax4 = plt.subplot(2, 2, 4, projection=projection)
ax4.set_extent([lon_min, lon_max, lat_min, lat_max], crs=projection)

# Add basic features
# ax4.add_feature(cfeature.COASTLINE, linewidth=0.8, edgecolor='black')

# Define raw coordinate grids specific to temperature data
raw_lats_temp = np.linspace(lat_min, lat_max, temp_raw_processed.shape[0])
raw_lons_temp = np.linspace(lon_min, lon_max, temp_raw_processed.shape[1])

# Plot raw temperature
im4 = ax4.contourf(raw_lons_temp, raw_lats_temp, temp_raw_processed, levels=20, cmap='jet',
                   transform=projection, extend='both')

# Add contour lines
contours4 = ax4.contour(raw_lons_temp, raw_lats_temp, temp_raw_processed, levels=10, colors='black',
                       linewidths=0.5, alpha=0.4, transform=projection)
ax4.clabel(contours4, inline=True, fontsize=7, fmt='%.1fK')

# Gridlines
gl4 = ax4.gridlines(draw_labels=True, dms=False, x_inline=False, y_inline=False,
                    linewidth=0.5, alpha=0.9, linestyle='--', color='gray')
gl4.top_labels = False
gl4.right_labels = False
gl4.xlabel_style = {'size': 12,'weight': 'bold'}
gl4.ylabel_style = {'size': 12, 'weight': 'bold'}

# Colorbar
cbar4 = plt.colorbar(im4, ax=ax4, orientation='vertical', pad=0.03, shrink=0.9)
cbar4.set_label('Temperature (K)', fontsize=11, fontweight='bold')
cbar4.ax.tick_params(labelsize=10)

# Make tick labels bold
for label in cbar4.ax.get_yticklabels():
    label.set_fontweight('bold')

# Statistics
# temp_stats_raw = f'Min: {np.nanmin(temp_raw_processed):.1f}K\nMax: {np.nanmax(temp_raw_processed):.1f}K\nMean: {np.nanmean(temp_raw_processed):.1f}K'
# props4 = dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor='black')
# ax4.text(0.02, 0.98, temp_stats_raw, transform=ax4.transAxes, fontsize=9,
#         verticalalignment='top', bbox=props4)

ax4.set_title('(d) Mean 2m Temperature (2020, ERA5-Land)', fontsize=13, fontweight='bold', pad=12)
plt.savefig('study_area.png', dpi=300, bbox_inches='tight')
plt.show()